In [1]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [2]:
brze_dim_carburant_df = spark.read.table("fuel_price_dev.bronze.brze_dim_carburant")
brze_dim_geo_df       = spark.read.table("fuel_price_dev.bronze.brze_dim_geo")
brze_dim_station_df   = spark.read.table("fuel_price_dev.bronze.brze_dim_station")
brze_fait_prix_df     = spark.read.table("fuel_price_dev.bronze.brze_fait_prix")
brze_fait_rupture_df  = spark.read.table("fuel_price_dev.bronze.brze_fait_rupture")

In [ ]:
slv_dim_carburant_df = brze_dim_carburant_df\
            .withColumn("date_traitement", now())\
            .withColumn("id", col("id").cast(IntegerType()))\
            .filter(col("id").isNotNull())\
            .select("id","nom","date_ingestion", "date_traitement")\
            .dropDuplicates(["id"])

slv_dim_geo_df = brze_dim_geo_df\
            .withColumn("date_traitement", now())\
            .filter(col("code_region").isNotNull() & col("code_departement").isNotNull() )\
            .select(
                "code_region",
                "region",
                "code_departement",
                "departement",
                "date_ingestion",
                "date_traitement"
        ).dropDuplicates(["code_region",])


slv_dim_station_df = brze_dim_station_df\
                      .withColumn("date_traitement", now())\
                      .withColumn("service", explode(split("service", ",")))\
                      .filter(col("station_id").isNotNull())\
                      .select(
                          "station_id",
                          "adresse",
                          "ville",
                          "code_postal",
                          "code_departement",
                          "service",
                          "code_region",
                          "date_ingestion",
                          "date_traitement"
                ).dropDuplicates(["station_id"])

In [0]:
# ============================================================================
# BRONZE FACTS  TRANSFORMATIONS
# ============================================================================
slv_fait_prix_df = brze_fait_prix_df\
                    .withColumn("date_traitement", now())\
                    .withColumn("carburant_id", col("carburant_id").cast(IntegerType()))\
                    .withColumn("date_maj", expr("try_cast(date_maj AS TIMESTAMP)"))\
                    .withColumn("prix", col("prix").cast(DoubleType()))\
                    .select("id_fct_pr",
                            "station_id",
                            "carburant_id",
                            "date_maj",
                            "prix",
                            "date_ingestion",
                            "date_traitement"
                    )\
                    .join(
                    slv_dim_station_df.select("station_id").distinct(),
                    on="station_id",
                    how="left_semi"
                )\
                .dropDuplicates(["id_fct_pr","date_maj"])
                        
slv_fait_rupture_df = brze_fait_rupture_df\
                        .withColumn("date_traitement", now())\
                        .withColumn("id_carburant", col("id_carburant").cast(IntegerType()))\
                        .withColumn("debut_rupture", expr("try_cast(debut_rupture AS TIMESTAMP)"))\
                        .withColumn("fin_rupture", expr("try_cast(fin_rupture AS TIMESTAMP)"))\
                        .select("id_fct_rpt",
                                "station_id",
                                "id_carburant",
                                "debut_rupture",
                                "fin_rupture",
                                "type_rupture",
                                "date_ingestion",
                                "date_traitement"
                        )\
                         .join(
                                slv_dim_station_df.select("station_id").distinct(),
                                on="station_id",
                                how="left_semi"
                        )\
                        .drop_duplicates(["id_fct_rpt","debut_rupture","fin_rupture"])


In [0]:
slv_dim_carburant_df.write\
                    .format("delta")\
                    .mode("overwrite")\
                    .option("overwriteSchema","true")\
                    .saveAsTable("fuel_price_dev.silver.dim_carburant")

slv_dim_geo_df.write\
                .format("delta")\
                .mode("overwrite")\
                .option("overwriteSchema","true")\
                .saveAsTable("fuel_price_dev.silver.dim_geo")  

slv_dim_station_df.write\
                    .format("delta")\
                    .mode("overwrite")\
                    .option("overwriteSchema","true")\
                    .saveAsTable("fuel_price_dev.silver.dim_station")   
                    
slv_fait_prix_df.write\
                    .format("delta")\
                    .mode("overwrite")\
                    .option("overwriteSchema","true")\
                    .saveAsTable("fuel_price_dev.silver.fait_prix")    

slv_fait_rupture_df.write\
                    .format("delta")\
                    .mode("overwrite")\
                    .option("overwriteSchema","true")\
                    .saveAsTable("fuel_price_dev.silver.fait_rupture")      
  

In [0]:
display(slv_dim_geo_df)

In [0]:
spark.sql("SELECT count(*) FROM fuel_price_dev.silver.dim_station s where s.code_departement NOT IN (SELECT dim_geo.code_departement  FROM fuel_price_dev.silver.dim_geo )")